# H3 · Mechanism: why is ROA lower in more developed countries?

> **Hypothesis H3:** The negative link between country development and company ROA is mostly explained by **what kind of companies** are listed and how they are financed (sector mix, size, leverage), not by country conditions as such.

| | |
|---|---|
| **Why we ask** | Notebook 03 found that firms in more developed countries have *lower* ROA (about 1.1 pp per SD). The sector table showed ROA depends heavily on the business model (banks hold huge assets, retailers turn assets over fast), and equity ratios fall where government debt is higher. A composition story is natural. |
| **Prediction** | The development coefficient **shrinks by at least half** (mediation share ≥ 50%) once sector, relative size and leverage are accounted for. |
| **Data** | Yahoo company-years (10,220). Same *development index* as in H2. Mediators: sector (12 groups), size rank, equity ratio (equity / assets, a leverage measure). Asset turnover and net margin are shown separately, because ROA = net margin × turnover mechanically. |
| **Primary test (fixed before looking at the result)** | Mediation share = 1 − (development coefficient with mediators) / (development coefficient with fiscal-year dummies only). 95% interval and p-value from a **country bootstrap** (resampling the 34 countries 999 times). |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. *H3 uses its own rule:* **supported** if the share is ≥ 50% and its 95% interval stays above 0; **partly supported** if the share is ≥ 25% with the interval above 0 (or ≥ 50% with the interval touching 0); otherwise **not supported**. If the starting link is itself not significant there is nothing to explain (*inconclusive*). |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra=""):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy), outcome=outcome, extra=extra, B=B)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

import patsy

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · The total association
First, the link to be explained: development index → ROA with only fiscal-year dummies (no sector or size controls), tested with the wild cluster bootstrap.

In [2]:
total = wild_cluster("dev_z", fy, outcome="roa_pp", controls="C(fiscal_year)", B=1999)
print(f"+1 SD development → {total['coef']:+.2f} pp ROA (95% CI {total['ci_low']:+.2f} to {total['ci_high']:+.2f}), wild p = {total['p_wild']:.4f}, {total['clusters']} countries, {total['n']:,} company-years")

+1 SD development → -1.12 pp ROA (95% CI -1.59 to -0.66), wild p = 0.0010, 34 countries, 10,220 company-years


## 2 · Peeling the effect: add one explanation at a time
Same sample in every row (companies with all the variables), so coefficients are comparable. If a layer explains part of the link, the development coefficient moves towards zero when it is added.

The last row, **net margin**, is *mechanical*: ROA is net income over assets, so net margin and asset turnover together almost define ROA. It is shown only to see which of the two moves with development.

In [3]:
need = ["roa_pp", "dev_z", "size_rank", "equity_ratio_w", "turnover_w", "net_margin_w"]
dd = fy.dropna(subset=need).copy()
specs = {"1. fiscal year only": "dev_z + C(fiscal_year)",
         "2. + sector": "dev_z + C(sector) + C(fiscal_year)",
         "3. + relative size": "dev_z + C(sector) + size_rank + C(fiscal_year)",
         "4. + leverage (equity ratio)": "dev_z + C(sector) + size_rank + equity_ratio_w + C(fiscal_year)",
         "5. + asset turnover": "dev_z + C(sector) + size_rank + equity_ratio_w + turnover_w + C(fiscal_year)",
         "6. + net margin (mechanical)": "dev_z + C(sector) + size_rank + equity_ratio_w + turnover_w + net_margin_w + C(fiscal_year)"}
rows = []
for lab, rhs in specs.items():
    m = fit_cluster("roa_pp ~ " + rhs, dd)
    ci = m.conf_int().loc["dev_z"]
    rows.append({"model": lab, "development coef (pp ROA per +1 SD)": m.params["dev_z"], "ci_low": ci[0], "ci_high": ci[1], "p (cluster-robust)": m.pvalues["dev_z"], "adj. R²": m.rsquared_adj})
peel = pd.DataFrame(rows).set_index("model")
peel["share of the first coefficient removed"] = 1 - peel.iloc[:, 0] / peel.iloc[0, 0]
print(f"n = {len(dd):,} company-years, {dd.iso3.nunique()} countries")
fig = px.bar(peel.reset_index(), x="development coef (pp ROA per +1 SD)", y="model", orientation="h", error_x=peel.ci_high - peel.iloc[:, 0], error_x_minus=peel.iloc[:, 0] - peel.ci_low,
             title="Development → ROA as explanations are added", height=420)
fig.add_vline(x=0, line_color="grey"); fig.update_layout(yaxis_autorange="reversed"); fig.show()
peel

n = 10,057 company-years, 34 countries


,development coef (pp ROA per +1 SD),ci_low,ci_high,p (cluster-robust),adj. R²,share of the first coefficient removed
model,,,,,,
1. fiscal year only,-0.990,-1.447,-0.533,0.000,0.015,0.000
2. + sector,-0.994,-1.456,-0.532,0.000,0.038,-0.005
3. + relative size,-0.997,-1.460,-0.535,0.000,0.038,-0.008
4. + leverage (equity ratio),-0.992,-1.436,-0.548,0.000,0.119,-0.002
5. + asset turnover,-0.971,-1.456,-0.485,0.000,0.155,0.019
6. + net margin (mechanical),-0.612,-1.012,-0.212,0.003,0.461,0.382


## 3 · Primary test: how much do sector, size and leverage explain?
Country bootstrap: resample the 34 countries with replacement, re-estimate the year-only and the composition models (steps 1 and 4), and record the share removed. The 95% interval shows how far this share can be trusted with only 34 countries.

In [4]:
ya, Xa = patsy.dmatrices("roa_pp ~ " + specs["1. fiscal year only"], dd, return_type="dataframe")
yd, Xd = patsy.dmatrices("roa_pp ~ " + specs["4. + leverage (equity ratio)"], dd, return_type="dataframe")
ya, yd = ya.values[:, 0], yd.values[:, 0]
ia, id_ = list(Xa.columns).index("dev_z"), list(Xd.columns).index("dev_z")
Xa, Xd = Xa.values, Xd.values
groups = {iso: np.flatnonzero(dd.iso3.values == iso) for iso in dd.iso3.unique()}
keys = list(groups)
rng = np.random.default_rng(0)
shares = []
for _ in range(999):
    idx = np.concatenate([groups[k] for k in rng.choice(keys, len(keys))])
    ba = np.linalg.lstsq(Xa[idx], ya[idx], rcond=None)[0][ia]
    bd = np.linalg.lstsq(Xd[idx], yd[idx], rcond=None)[0][id_]
    shares.append(1 - bd / ba if ba < 0 else np.nan)          # nothing to explain if the starting link is not negative
shares = np.array(shares)
ok = ~np.isnan(shares)
share = float(peel.loc["4. + leverage (equity ratio)", "share of the first coefficient removed"])
lo, hi = np.nanpercentile(shares, [2.5, 97.5])
p_boot = float((np.sum(shares[ok] <= 0) + 1) / (ok.sum() + 1))
print(f"Share of the development–ROA link explained by sector + size + leverage: {share:.0%} (95% interval {lo:.0%} to {hi:.0%}); one-sided bootstrap p (share ≤ 0) = {p_boot:.3f}; {ok.sum()} of 999 draws had a negative starting link.")
px.histogram(x=np.clip(shares[ok], -1, 2), nbins=50, title="Country-bootstrap distribution of the share explained (clipped to −100%…200%)", labels={"x": "share explained"}, height=360).add_vline(x=share, line_color="red").add_vline(x=0, line_color="grey")

Share of the development–ROA link explained by sector + size + leverage: -0% (95% interval -36% to 20%); one-sided bootstrap p (share ≤ 0) = 0.500; 999 of 999 draws had a negative starting link.


## 4 · Which channel? Margin or asset use?
Outcomes regressed on the development index, with sector, size and year controls (errors clustered by country). **Net margin** = net income / revenue; **asset turnover** = revenue / assets; **equity ratio** = equity / assets; **financial** = a financial-sector company (0/1, in pp).

In [5]:
fy["financial_pp"] = fy.is_financial * 100
chan = {"net margin (pp)": "net_margin_w", "asset turnover (×)": "turnover_w", "equity ratio (×)": "equity_ratio_w", "ROA (pp)": "roa_pp"}
rows = []
for lab, col in chan.items():
    d = fy.dropna(subset=[col, "size_rank", "dev_z"]).copy()
    sc = 100 if "pp" in lab and col != "roa_pp" else 1
    d[col] = d[col] * sc
    m = fit_cluster(f"{col} ~ dev_z + {CONTROLS}", d); ci = m.conf_int().loc["dev_z"]
    rows.append({"outcome": lab, "coef per +1 SD development": m.params["dev_z"], "ci_low": ci[0], "ci_high": ci[1], "p": m.pvalues["dev_z"], "median of outcome": d[col].median()})
d = fy.dropna(subset=["size_rank", "dev_z"]); m = fit_cluster("financial_pp ~ dev_z + size_rank + C(fiscal_year)", d); ci = m.conf_int().loc["dev_z"]
rows.append({"outcome": "share financial-sector (pp)", "coef per +1 SD development": m.params["dev_z"], "ci_low": ci[0], "ci_high": ci[1], "p": m.pvalues["dev_z"], "median of outcome": fy.financial_pp.mean()})
channels = pd.DataFrame(rows).set_index("outcome")
channels

,coef per +1 SD development,ci_low,ci_high,p,median of outcome
outcome,,,,,
net margin (pp),-3.012,-4.564,-1.460,0.000,8.078
asset turnover (×),-0.010,-0.051,0.031,0.628,0.541
equity ratio (×),-0.000,-0.014,0.014,0.990,0.419
ROA (pp),-1.124,-1.599,-0.648,0.000,3.777
share financial-sector (pp),-0.138,-1.028,0.751,0.760,11.438


**Within each sector:** is the negative development–ROA link a general one, or driven by a few sectors (for example banks)?

In [6]:
rows = []
for sct, d in fy.dropna(subset=["roa_pp", "dev_z", "size_rank"]).groupby("sector"):
    if d.iso3.nunique() < 10:
        continue
    m = fit_cluster("roa_pp ~ dev_z + size_rank + C(fiscal_year)", d); ci = m.conf_int().loc["dev_z"]
    rows.append({"sector": sct, "coef": m.params["dev_z"], "ci_low": ci[0], "ci_high": ci[1], "countries": d.iso3.nunique(), "firm-years": len(d)})
bysec = pd.DataFrame(rows).sort_values("coef")
fig = px.scatter(bysec, x="coef", y="sector", error_x=bysec.ci_high - bysec.coef, error_x_minus=bysec.coef - bysec.ci_low, hover_data=["countries", "firm-years"],
                 title="Development → ROA within each sector (pp per +1 SD; cluster-robust 95% CI)")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=460); fig.show()
print(f"Negative in {(bysec.coef < 0).sum()} of {len(bysec)} sectors; CI excludes zero in {((bysec.ci_high < 0) | (bysec.ci_low > 0)).sum()}.")

Negative in 10 of 11 sectors; CI excludes zero in 5.


## 5 · Robustness
The mediation share (steps 1 → 4) re-estimated in the standard samples (point estimates only; the bootstrap above gives the uncertainty), and the development link itself in each sample.

In [7]:
rows = []
for name, flt in SAMPLES.items():
    d = flt(dd)
    try:
        ma = fit_cluster("roa_pp ~ " + specs["1. fiscal year only"], d).params["dev_z"]
        md_ = fit_cluster("roa_pp ~ " + specs["4. + leverage (equity ratio)"], d).params["dev_z"]
        rows.append({"sample": name, "link (yr only)": ma, "link (+ sector, size, leverage)": md_, "share explained": 1 - md_ / ma if ma < 0 else np.nan, "countries": d.iso3.nunique()})
    except Exception:
        pass
rob = pd.DataFrame(rows).set_index("sample")
rob_share = float(((rob["share explained"] >= 0.25) & (rob["link (yr only)"] < 0)).mean())
print(f"Share explained ≥ 25% (with a negative starting link) in {rob_share:.0%} of the samples.")
rob

Share explained ≥ 25% (with a negative starting link) in 0% of the samples.


,link (yr only),"link (+ sector, size, leverage)",share explained,countries
sample,,,,
all firms,-0.990,-0.992,-0.002,34
excluding financials & real estate,-1.118,-1.135,-0.015,34
excluding small-sample countries,-0.974,-0.977,-0.003,29
"excluding USA, China, Japan",-1.096,-1.106,-0.009,31
fiscal years 2023 and later,-1.045,-1.037,0.008,34
EU countries only,-1.017,-1.121,-0.103,21
non-EU countries only,-0.656,-0.639,0.027,13


## 6 · Replication on ESEF (EU, no sector)
Here only size and leverage can be accounted for, so the share is a partial measure.

In [8]:
fe = prep.load_firms(con, country, source="ESEF")
de = fe.dropna(subset=["roa_pp", "dev_z", "size_rank", "equity_ratio_w"]).copy()
ma = fit_cluster("roa_pp ~ dev_z + C(fiscal_year)", de); mb = fit_cluster("roa_pp ~ dev_z + size_rank + equity_ratio_w + C(fiscal_year)", de)
rep_total = wild_cluster("dev_z", fe, outcome="roa_pp", controls="C(fiscal_year)", B=999)
rep_share = 1 - mb.params["dev_z"] / ma.params["dev_z"]
print(f"ESEF ({len(de):,} company-years, {de.iso3.nunique()} countries): development → ROA = {ma.params['dev_z']:+.2f} pp (wild p = {rep_total['p_wild']:.3f}); after size + leverage {mb.params['dev_z']:+.2f} pp → share removed {rep_share:.0%}.")

ESEF (5,247 company-years, 19 countries): development → ROA = -1.34 pp (wild p = 0.021); after size + leverage -1.51 pp → share removed -13%.


## 7 · Verdict

In [9]:
if total["p_wild"] >= 0.05 or total["coef"] >= 0:
    v = "inconclusive"
elif share >= 0.5 and lo > 0:
    v = "supported"
elif (share >= 0.25 and lo > 0) or share >= 0.5:
    v = "partly supported"
else:
    v = "not supported"
pct = lambda x: f"{round(100 * x):d}%"
margin_channel = channels.loc["net margin (pp)", "p"] < 0.05 and channels.loc["asset turnover (×)", "p"] > 0.10
lines = [
 f"Starting link: +1 SD development → {total['coef']:+.2f} pp ROA (wild p = {total['p_wild']:.3f}).",
 f"Sector + size + leverage remove {pct(share)} of it (95% interval {pct(lo)} to {pct(hi)}, bootstrap p = {p_boot:.3f}). Sector alone: {pct(peel.iloc[1, -1])}; adding size: {pct(peel.iloc[2, -1])}; adding leverage: {pct(peel.iloc[3, -1])}.",
 f"Channels (with controls): margin {channels.loc['net margin (pp)', 'coef per +1 SD development']:+.2f} pp (p = {channels.loc['net margin (pp)', 'p']:.3f}), asset turnover {channels.loc['asset turnover (×)', 'coef per +1 SD development']:+.2f} (p = {channels.loc['asset turnover (×)', 'p']:.3f}), equity ratio {channels.loc['equity ratio (×)', 'coef per +1 SD development']:+.3f} (p = {channels.loc['equity ratio (×)', 'p']:.3f}).",
 f"Within sectors the link is negative in {(bysec.coef < 0).sum()} of {len(bysec)} sectors. ESEF: share removed by size + leverage {pct(rep_share)}.",
 ("What the data point to instead: the link works through thinner NET MARGINS within sectors, not through asset use or leverage." if margin_channel else "The channel is not clear from margin versus asset use."),
 f"VERDICT H3: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H3", id="H3", title="Mechanism of the development–ROA link", statement="Sector mix, size and leverage explain most of the negative link between development and ROA.",
                  expected_sign=1, effect=share, ci_low=float(lo), ci_high=float(hi), effect_unit="share of the development–ROA link explained",
                  p_primary=p_boot, p_method="country bootstrap (share ≤ 0)", n=int(len(dd)), clusters=int(dd.iso3.nunique()), robust_share=rob_share,
                  replication_coef=float(rep_share), replication_p=float(rep_total["p_wild"]), replication_same_sign=bool(rep_share > 0), verdict=v,
                  notebook="1.3_mechanism_of_the_roa_link.ipynb", notes=lines)

Starting link: +1 SD development → -1.12 pp ROA (wild p = 0.001).
Sector + size + leverage remove 0% of it (95% interval -36% to 20%, bootstrap p = 0.500). Sector alone: 0%; adding size: -1%; adding leverage: 0%.
Channels (with controls): margin -3.01 pp (p = 0.000), asset turnover -0.01 (p = 0.628), equity ratio -0.000 (p = 0.990).
Within sectors the link is negative in 10 of 11 sectors. ESEF: share removed by size + leverage -13%.
What the data point to instead: the link works through thinner NET MARGINS within sectors, not through asset use or leverage.
VERDICT H3: NOT SUPPORTED


### Limits
- A mediation analysis assumes the mediators (sector, leverage) are *not themselves caused* by development in ways that matter here; in reality listing structure and leverage are partly outcomes of a country's development, so "explained by composition" is a description of where the link comes from, not a causal chain.
- With 34 countries the interval for the share is wide.
- Net margin and asset turnover are mechanically tied to ROA; the channel table is descriptive.
- Association, not causation.

In [10]:
con.close()